# Part 4 — GenAI: Structured Follow-up Note Analysis

Tasks 4.1–4.4. This notebook requires a free Gemini API key entered at runtime. **Never paste the key into a code cell, notebook text, or GitHub.** Clear notebook outputs before committing. The first 10 call-centre notes are analysed; blank note N004 is skipped without an API call.

In [ ]:
!pip install -q google-genai pydantic

import json
import time
import pandas as pd
from pathlib import Path
from pydantic import BaseModel, Field, ValidationError
from google import genai
from google.genai import types

import os
import subprocess
REPO_URL = 'https://github.com/Subham9339063436/CareLoop-Clinics-No-Show-Guard.git'
REPO_DIR = Path('/content/CareLoop-Clinics-No-Show-Guard')
if not (Path.cwd() / 'data' / 'followup_notes.csv').exists():
    if not (REPO_DIR / 'data' / 'followup_notes.csv').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

ROOT = Path.cwd()
NOTES_FILE = ROOT / 'data' / 'followup_notes.csv'
APPOINTMENTS_FILE = ROOT / 'appointments_clean.csv'
MODEL_NAME = 'gemini-3.8-flash'  # Current stable Flash model; change only if unavailable in your AI Studio account.
API_KEY = input('Paste your Gemini API key (runtime only): ')
client = genai.Client(api_key=API_KEY)
API_CALLS = 0
ALLOWED_ROOT_CAUSES = [
    'forgot_or_no_reminder', 'distance_or_transport', 'cost_or_fee',
    'illness_or_emergency', 'work_or_schedule', 'other'
]

SCHEMA = {
    'type': 'object',
    'properties': {
        'root_cause': {'type': 'string'},
        'confidence': {'type': 'number'},
        'summary': {'type': 'string'},
        'recommended_action': {'type': 'string'}
    },
    'required': ['root_cause', 'confidence', 'summary', 'recommended_action']
}

SYSTEM_TEXT = (
    'You are a careful outpatient-clinic call-centre note analyst. '
    'Classify only what the note supports. Do not infer medical details or invent facts. '
    'Use other when the cause is ambiguous or does not fit a listed category.'
)


## Task 4.1 — One validated call: N001

The CRAFT prompt supplies the context, role, action, tone and required output format. The schema keeps `root_cause` as a plain string; the allowed-value rule is enforced afterward by Pydantic/application validation.

In [ ]:
def build_craft_prompt(note_text):
    return f"""Context: CareLoop Clinics is reviewing short call-centre notes recorded after a patient missed an outpatient appointment.
Role: You are a careful clinical-administration data analyst, not a clinician.
Action: Classify the main reason supported by the note. Choose exactly one root_cause from this list: {', '.join(ALLOWED_ROOT_CAUSES)}. Use 'other' rather than guessing when the note is unclear or does not support a listed reason. Keep summary to at most 25 words. Recommend a practical administrative follow-up, not medical advice.
Tone: Neutral, concise, factual, non-judgmental.
Format: Return only the required JSON object with exactly these fields: root_cause (string), confidence (number from 0 to 1), summary (string, at most 25 words), recommended_action (string).

Note to classify:
{note_text}"""


def generate_json(prompt):
    global API_CALLS
    # Gemini 3.8 Flash supports low/medium/high thinking; minimal is not supported.
    # The assignment allows removing thinking_config if a model rejects the setting.
    base_config = {
        'system_instruction': SYSTEM_TEXT,
        'temperature': 0,
        'max_output_tokens': 1000,
        'response_mime_type': 'application/json',
        'response_schema': SCHEMA
    }
    config = types.GenerateContentConfig(
        **base_config,
        thinking_config=types.ThinkingConfig(thinking_level='low')
    )
    try:
        API_CALLS += 1
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=config
        )
    except Exception as exc:
        if 'thinking_config' not in str(exc).lower() and 'thinking' not in str(exc).lower():
            raise
        fallback_config = types.GenerateContentConfig(**base_config)
        API_CALLS += 1
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=fallback_config
        )
    return response.text or '' 


def analyse_note(note_text):
    raw_text = generate_json(build_craft_prompt(note_text))
    return json.loads(raw_text)

n001 = pd.read_csv(NOTES_FILE).loc[lambda d: d['note_id'].eq('N001'), 'note_text'].iloc[0]
parsed_n001 = analyse_note(n001)
print(parsed_n001)
assert set(parsed_n001) == {'root_cause', 'confidence', 'summary', 'recommended_action'}


## Task 4.2 — Pydantic validation and process first 10 notes

In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)


def validate_allowed_root_cause(data):
    if data.root_cause not in ALLOWED_ROOT_CAUSES:
        raise ValueError(f'root_cause must be one of {ALLOWED_ROOT_CAUSES}; received {data.root_cause!r}')
    return data

notes_df = pd.read_csv(NOTES_FILE).head(10)
calls_before_task_42 = API_CALLS
analysed_records = []
for row_number, row in notes_df.iterrows():
    note_text = row['note_text']
    base = {
        'note_id': row['note_id'],
        'appointment_id': row['appointment_id'],
        'root_cause': '', 'confidence': '', 'summary': '',
        'recommended_action': '', 'validation_status': '', 'raw_response': ''
    }
    # Check pandas NaN before str(...).strip(), because NaN is a float.
    if pd.isna(note_text) or str(note_text).strip() == '':
        base.update({
            'root_cause': 'other', 'confidence': 0.0,
            'summary': 'No note recorded',
            'recommended_action': 'Re-call patient',
            'validation_status': 'skipped_blank', 'raw_response': ''
        })
        analysed_records.append(base)
        print(row['note_id'], 'blank note: skipped API call')
        continue

    raw_text = ''
    response_data = {}
    try:
        raw_text = generate_json(build_craft_prompt(str(note_text)))
        response_data = json.loads(raw_text)
        if not isinstance(response_data, dict):
            raise ValueError('Model response JSON must be an object')
        # Copy only fields actually returned; do not repair missing/invalid content.
        for key in ['root_cause', 'confidence', 'summary', 'recommended_action']:
            if key in response_data:
                base[key] = response_data[key]
        validated = FollowupInsight(**response_data)
        validate_allowed_root_cause(validated)
        base['validation_status'] = 'valid'
    except Exception as exc:
        base['validation_status'] = 'invalid'
        print(row['note_id'], 'reply retained as invalid:', str(exc))
    base['raw_response'] = raw_text
    analysed_records.append(base)
    time.sleep(13)  # Free-tier rate-limit pause between nonblank calls.

notes_analysed = pd.DataFrame(analysed_records, columns=[
    'note_id', 'appointment_id', 'root_cause', 'confidence', 'summary',
    'recommended_action', 'validation_status', 'raw_response'
])
notes_analysed.to_csv(ROOT / 'notes_analysed.csv', index=False)
print(notes_analysed[['note_id', 'root_cause', 'confidence', 'validation_status']].to_string(index=False))
print('Part 4.2 API calls:', API_CALLS - calls_before_task_42)


## Task 4.3 — Deliberately invalid replies and prompt comparison

In [ ]:
bad_replies = [
    {
        'root_cause': 'cost_or_fee', 'confidence': 1.7,
        'summary': 'Fee was too high', 'recommended_action': 'Offer fee information'
    },
    {
        'root_cause': 'work_or_schedule', 'confidence': 0.8,
        'summary': 'Work conflict was reported'
        # recommended_action intentionally missing
    }
]
for number, bad_reply in enumerate(bad_replies, start=1):
    try:
        FollowupInsight(**bad_reply)
    except ValidationError as exc:
        print(f'Bad reply {number} validation error:\n{exc}\n')

# V1: intentionally minimal prompt. V2: reuse the CRAFT replies already generated in Task 4.2.
def get_raw_root_cause(raw_text):
    try:
        parsed = json.loads(raw_text)
        value = parsed.get('root_cause', '') if isinstance(parsed, dict) else ''
        return value, value in ALLOWED_ROOT_CAUSES
    except Exception:
        return '', False

ab_rows = []
for note_id in ['N007', 'N009']:
    note_text = notes_df.loc[notes_df['note_id'].eq(note_id), 'note_text'].iloc[0]
    v1_prompt = 'Classify this note. Return JSON matching the required schema.\n\n' + str(note_text)
    v1_raw = generate_json(v1_prompt)
    v1_cause, v1_allowed = get_raw_root_cause(v1_raw)

    v2_row = notes_analysed.loc[notes_analysed['note_id'].eq(note_id)].iloc[0]
    v2_cause, v2_allowed = get_raw_root_cause(v2_row['raw_response'])
    ab_rows.append({'note_id': note_id, 'prompt_version': 'V1 basic', 'root_cause': v1_cause, 'allowed_value': v1_allowed})
    ab_rows.append({'note_id': note_id, 'prompt_version': 'V2 CRAFT (reused Task 4.2)', 'root_cause': v2_cause, 'allowed_value': v2_allowed})

prompt_comparison = pd.DataFrame(ab_rows)
print(prompt_comparison.to_string(index=False))
print('Deploy V2 CRAFT: it specifies the allowed categories and requires other rather than guessing when the note is ambiguous.')
print('Total API calls across Tasks 4.1–4.3:', API_CALLS)


## Task 4.4 — Merge notes with appointment data and write recommendation evidence

In [ ]:
appointments_clean = pd.read_csv(APPOINTMENTS_FILE)
merged_notes = notes_analysed.merge(
    appointments_clean[['appointment_id', 'reminder_sent', 'no_show', 'clinic_id', 'lead_days']],
    on='appointment_id', how='left', validate='one_to_one'
)
print('Merged note rows:', len(merged_notes))
print('Root cause by reminder status:\n', pd.crosstab(merged_notes['root_cause'], merged_notes['reminder_sent']))
print('Notes linked to reminder_sent=No:', int((merged_notes['reminder_sent'] == 'No').sum()), 'of', len(merged_notes))
print('Final note output file:', ROOT / 'notes_analysed.csv')


## Submission and privacy check

The final notebook must not contain the API key. The key is entered via `input()` at runtime. Clear outputs before committing if any output could contain the key. Commit the generated `notes_analysed.csv` after the run and copy the measured Part 4 counts and crosstab into `README.md`.